## Requirement 1 - merging the channel features with the transactional sales

The merge is resolved here rather than against the fact table.
`trade_group` and `trade_type` are attributes of the channel, so they are
joined once while the dimension is built.

In [0]:
from pyspark.sql import functions as F

In [0]:
# silver sales
SOURCE_CATALOG_NAME_SS = 'beverage_sales'
SOURCE_SCHEMA_NAME_SS = 'silver'
SOURCE_TABLE_NAME_SS = 'sales'

# silver channel group
SOURCE_CATALOG_NAME_CG = 'beverage_sales'
SOURCE_SCHEMA_NAME_CG = 'silver'
SOURCE_TABLE_NAME_CG = 'channel_group'

# dim_channel
TARGET_CATALOG_NAME = 'beverage_sales'
TARGET_SCHEMA_NAME = 'gold'
TARGET_TABLE_NAME = 'dim_channel'

UNKNOWN_KEY = -1

In [0]:
df_silver_sales = spark.table(f'{SOURCE_CATALOG_NAME_SS}.{SOURCE_SCHEMA_NAME_SS}.{SOURCE_TABLE_NAME_SS}')
df_channel_group = spark.table(f'{SOURCE_CATALOG_NAME_CG}.{SOURCE_SCHEMA_NAME_CG}.{SOURCE_TABLE_NAME_CG}')

In [0]:
df_unknown_member = spark.createDataFrame(
    [(UNKNOWN_KEY, 'UNKNOWN', 'UNKNOWN', 'UNKNOWN', 'UNKNOWN')],
    'channel_key bigint, trade_channel string, channel_group string, '
    'trade_group string, trade_type string'
)

## Cardinality check

`dropDuplicates` on a natural key is only safe when that key functionally
determines the remaining attributes. A future file where one `trade_channel` carries two different
attribute sets fails here instead of silently keeping an arbitrary row.

In [0]:
df_invalid_mapping = (
    df_silver_sales
    .groupBy('trade_channel')
    .agg(F.countDistinct('channel_group').alias('attribute_count'))
    .filter(F.col('attribute_count') > 1)
)

assert df_invalid_mapping.count() == 0, 'trade_channel does not uniquely determine channel_group'

In [0]:
df_dim_channel = (
    df_silver_sales
    .select('trade_channel', 'channel_group')
    .dropDuplicates(['trade_channel'])
    .join(
        F.broadcast(df_channel_group),
        on='trade_channel',
        how='left'
    )
    .withColumn('channel_key', F.abs(F.xxhash64(F.col('trade_channel'))))
    .withColumn('trade_group', F.coalesce(F.col('trade_group'), F.lit('UNMAPPED')))
    .withColumn('trade_type', F.coalesce(F.col('trade_type'), F.lit('UNMAPPED')))
    .select(
        'channel_key',
        'trade_channel',
        'channel_group',
        'trade_group',
        'trade_type'
    )
    .unionByName(df_unknown_member)
)

In [0]:
df_dim_channel\
    .write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')